# LLMs for Text Transformation

In this notebook, we will explore how to use Large Language Models for text transformation tasks such as language translation, spelling and grammar checking, tone adjustment, and format conversion.

## Setup

In [1]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, 
    )
    return response.choices[0].message.content

## Translation

ChatGPT is trained with sources in many languages. This gives the model the ability to do translation. Here are some examples of how to use this capability.

In [3]:
prompt = f"""
Translate the following English text to Spanish: \ 
```Hi, I would like to order a blender```
"""
response = get_completion(prompt)
print(response)

Hola, me gustaría ordenar una licuadora.


In [4]:
prompt = f"""
Tell me which language this is: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)

This is French.


In [5]:
prompt = f"""
Translate the following  text to French and Spanish
and English pirate: \
```I want to order a basketball```
"""
response = get_completion(prompt)
print(response)

French: Je veux commander un ballon de basket
Spanish: Quiero ordenar un balón de baloncesto
English: I want to order a basketball


In [6]:
prompt = f"""
Translate the following text to Spanish in both the \
formal and informal forms: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)

Formal: ¿Le gustaría ordenar una almohada?
Informal: ¿Te gustaría ordenar una almohada?


### Universal Translator
Imagine you are in charge of IT at a large multinational e-commerce company. Users are messaging you with IT issues in all their native languages. Your staff is from all over the world and speaks only their native languages. You need a universal translator!

In [7]:
user_messages = [
  "La performance du système est plus lente que d'habitude.",  # System performance is slower than normal         
  "Mi monitor tiene píxeles que no se iluminan.",              # My monitor has pixels that are not lighting
  "Il mio mouse non funziona",                                 # My mouse is not working
  "Mój klawisz Ctrl jest zepsuty",                             # My keyboard has a broken control key
  "我的屏幕在闪烁"                                               # My screen is flashing
] 

In [8]:
for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"Original message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English \
    and Korean: ```{issue}```
    """
    response = get_completion(prompt)
    print(response, "\n")

Original message (French): La performance du système est plus lente que d'habitude.
English: "The system performance is slower than usual."

Korean: "시스템 성능이 평소보다 느립니다." 

Original message (This is Spanish.): Mi monitor tiene píxeles que no se iluminan.
English: "My monitor has pixels that do not light up."
Korean: "내 모니터에는 불이 켜지지 않는 픽셀이 있습니다." 

Original message (Italian): Il mio mouse non funziona
English: My mouse is not working
Korean: 내 마우스가 작동하지 않습니다 

Original message (This is Polish.): Mój klawisz Ctrl jest zepsuty
English: My Ctrl key is broken
Korean: 제 Ctrl 키가 고장 났어요 

Original message (This is Chinese.): 我的屏幕在闪烁
English: My screen is flickering
Korean: 내 화면이 깜박거립니다 



## Tone Transformation
Writing can vary based on the intended audience. ChatGPT can produce different tones.


In [9]:
prompt = f"""
Translate the following from slang to a business letter: 
'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response = get_completion(prompt)
print(response)

Dear Sir/Madam,

I am writing to bring to your attention the specifications of the standing lamp. 

Sincerely,
Joe


## Format Conversion
ChatGPT can translate between formats. The prompt should describe the input and output formats.

In [11]:
data_json = { "resturant employees" :[ 
    {"name":"Shyam", "email":"shyamjaiswal@gmail.com"},
    {"name":"Bob", "email":"bob32@gmail.com"},
    {"name":"Jai", "email":"jai87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_completion(prompt)
print(response)

<html>
<head>
  <title>Restaurant Employees</title>
</head>
<body>
  <table>
    <tr>
      <th>Name</th>
      <th>Email</th>
    </tr>
    <tr>
      <td>Shyam</td>
      <td>shyamjaiswal@gmail.com</td>
    </tr>
    <tr>
      <td>Bob</td>
      <td>bob32@gmail.com</td>
    </tr>
    <tr>
      <td>Jai</td>
      <td>jai87@gmail.com</td>
    </tr>
  </table>
</body>
</html>


In [12]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))

Name,Email
Shyam,shyamjaiswal@gmail.com
Bob,bob32@gmail.com
Jai,jai87@gmail.com


## Spellcheck/Grammar check.

Here are some examples of common grammar and spelling problems and the LLM's response. 

To signal to the LLM that you want it to proofread your text, you instruct the model to 'proofread' or 'proofread and correct'.

In [13]:
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]
for t in text:
    prompt = f"""Proofread and correct the following text
    and rewrite the corrected version. If you don't find
    and errors, just say "No errors found". Don't use 
    any punctuation around the text:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

The girl with the black and white puppies has a ball.
No errors found
No errors found
Their goes my freedom. There going to bring they’re suitcases.

No errors found

Rewritten:
Their goes my freedom. There going to bring their suitcases.
You're going to need your notebook.
No errors found.
No errors found


In [14]:
text = f"""
Got this for my daughter for her birthday cuz she keeps taking \
mine from my room.  Yes, adults also like pandas too.  She takes \
it everywhere with her, and it's super soft and cute.  One of the \
ears is a bit lower than the other, and I don't think that was \
designed to be asymmetrical. It's a bit small for what I paid for it \
though. I think there might be other options that are bigger for \
the same price.  It arrived a day earlier than expected, so I got \
to play with it myself before I gave it to my daughter.
"""
prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
print(response)

I got this for my daughter for her birthday because she keeps taking mine from my room. Yes, adults also like pandas too. She takes it everywhere with her, and it's super soft and cute. One of the ears is a bit lower than the other, and I don't think that was designed to be asymmetrical. It's a bit small for what I paid for it though. I think there might be other options that are bigger for the same price. It arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.


In [ ]:
#! pip3 install redlines


   ------------- -------------------------- 1/3 [rich-click]
   ---------------------------------------- 3/3 [redlines]



In [16]:
from redlines import Redlines

diff = Redlines(text,response)
display(Markdown(diff.output_markdown))

<span style='color:red;font-weight:700;text-decoration:line-through;'>Got </span><span style='color:green;font-weight:700;'>I got </span>this for my daughter for her birthday <span style='color:red;font-weight:700;text-decoration:line-through;'>cuz </span><span style='color:green;font-weight:700;'>because </span>she keeps taking mine from my room.  Yes, adults also like pandas too.  She takes it everywhere with her, and it's super soft and cute.  One of the ears is a bit lower than the other, and I don't think that was designed to be asymmetrical. It's a bit small for what I paid for it though. I think there might be other options that are bigger for the same price.  It arrived a day earlier than expected, so I got to play with it myself before I gave it to my daughter.

In [17]:
prompt = f"""
proofread and correct this review. Make it more compelling. 
Ensure it follows APA style guide and targets an advanced reader. 
Output in markdown format.
Text: ```{text}```
"""
response = get_completion(prompt)
display(Markdown(response))

I purchased this adorable panda plush for my daughter's birthday as she kept borrowing mine from my room. It's not just kids who love pandas - adults do too! She carries it everywhere, and it's incredibly soft and charming. However, I did notice that one ear is slightly lower than the other, which seems unintentional. Despite its cuteness, I found it to be smaller than expected given the price. I believe there are larger options available for the same cost. On a positive note, it arrived a day earlier than anticipated, allowing me to enjoy it briefly before gifting it to my daughter. Overall, while the panda is delightful, there may be better alternatives in terms of size and design for the price point.

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [19]:
text = f"""
The artifact, dubbed the Stargate, turned out to be an alien device capable of \
opening stable wormholes for instantaneous travel to other planets. On their \
first official mission, Jackson and the strict Colonel Jack O'Neill crossed the \
threshold toward the desert planet Abydos. There, they discovered that the \
ancient gods of Egyptian mythology were actually the Goa'uld: a race of \
snake-like alien parasites that took human bodies as hosts to pose as national \
deities and enslave humanity across the galaxy.
"""
prompt = f"Transalte this text to old spanish, like Cervantes 'Don Quixote': ```{text}```"
response = get_completion(prompt)
print(response)

El artefacto, llamado la Puerta Estelar, resultó ser un dispositivo alienígena capaz de abrir agujeros de gusano estables para viajar instantáneamente a otros planetas. En su primera misión oficial, Jackson y el estricto Coronel Jack O'Neill cruzaron el umbral hacia el planeta desértico de Abydos. Allí descubrieron que los antiguos dioses de la mitología egipcia eran en realidad los Goa'uld: una raza de parásitos alienígenas parecidos a serpientes que tomaban cuerpos humanos como anfitriones para hacerse pasar por deidades nacionales y esclavizar a la humanidad en toda la galaxia.


In [20]:
text = f"""
The artifact, dubbed the Stargate, turned out to be an alien device capable of \
opening stable wormholes for instantaneous travel to other planets. On their \
first official mission, Jackson and the strict Colonel Jack O'Neill crossed the \
threshold toward the desert planet Abydos. There, they discovered that the \
ancient gods of Egyptian mythology were actually the Goa'uld: a race of \
snake-like alien parasites that took human bodies as hosts to pose as national \
deities and enslave humanity across the galaxy.
"""
prompt = f"Generate an ASCII art image of what the Stargate would look like based on this text: ```{text}```"
response = get_completion(prompt)
print(response)

```
    _______  _______  _______  _______  _______  _______  _______  _______  _______ 
   (  ____ \(  ___  )(  ____ \(  ____ \(  ____ \(  ____ \(  ____ \(  ____ \(  ____ \
   | (    \/| (   ) || (    \/| (    \/| (    \/| (    \/| (    \/| (    \/| (    \/
   | (__    | (___) || (__    | (____  | (____  | (____  | (____  | (____  | (____  
   |  __)   |  ___  ||  __)   (_____ \ (_____ \ (_____ \ (_____ \ (_____ \ (_____ \
   | (      | (   ) || (            ) )       ) )       ) )       ) )       ) )
   | (____/\| )   ( || (____/\/\____) )/\____) )/\____) )/\____) )/\____) )/\____) )
   (_______/|/     \|(_______/\______/ \______/ \______/ \______/ \______/ \______/
```


In [21]:
text = f"""
The artifact, dubbed the Stargate, turned out to be an alien device capable of \
opening stable wormholes for instantaneous travel to other planets. On their \
first official mission, Jackson and the strict Colonel Jack O'Neill crossed the \
threshold toward the desert planet Abydos. There, they discovered that the \
ancient gods of Egyptian mythology were actually the Goa'uld: a race of \
snake-like alien parasites that took human bodies as hosts to pose as national \
deities and enslave humanity across the galaxy.
"""
prompt = f"Write the text as if you were speaking from the perspective of the main character, Colonel Jack O'Neill: ```{text}```"
response = get_completion(prompt)
print(response)

So there we were, Jackson and I, stepping through that Stargate for the first time. The moment we arrived on Abydos, everything changed. We learned the truth about the Goa'uld, these so-called gods that had been manipulating civilizations for centuries. It was a lot to take in, but we knew we had to do something about it. And that's when our real journey began - fighting against these powerful beings to protect Earth and all of humanity.


- In the first iteration, we asked the model to translate a passage taken from a public domain book written in Old Spanish. As we can see, it provides a good interpretation, most likely because the model was trained on this work or on similar texts from the same 16th-century author, allowing it to understand the language and style accurately.

- In the second iteration, we requested a task based on the generated description: formatting an ASCII representation using the context of an element mentioned in the text. As shown, the result bears no resemblance to the well-known Stargate, since the contextual description is not sufficiently precise. Nevertheless, the model correctly associates the requested output format and generates a reasonable result, even though it is effectively hallucinating the specific details.

- In the third iteration, we asked the model to describe the text from the perspective of one of the characters appearing in it. The resulting output closely resembles what that character might have said when recounting their own experience. This demonstrates that the model is capable of inferring the different elements of the described scene and combining them into a coherent narrative.

LLMs are capable of producing outputs in many different formats depending on their training and the context provided. The quality and appropriateness of the result largely depend on the amount of contextual information available and the complexity of the requested task.